In [ ]:
import os

import numpy as np
import matplotlib.pyplot as plt
from netin.models import PATCHModel, CompoundLFM

from patch.constants import PATH_INFERENCE, MAP_LFM_SHORT

In [ ]:
PATH_PREFIX = "ccf-mean_"
LFM_COMBS = [
    (CompoundLFM.HOMOPHILY.value, CompoundLFM.UNIFORM.value),
    (CompoundLFM.HOMOPHILY.value, CompoundLFM.HOMOPHILY.value),
    (CompoundLFM.PAH.value, CompoundLFM.UNIFORM.value),
    (CompoundLFM.PAH.value, CompoundLFM.PAH.value),
]
DECADE = 1980

In [ ]:
def create_folder_name(lfm_global: str, lfm_tc: str, decade: int = DECADE, prefix: str = PATH_PREFIX) -> str:
    return os.path.join(
        "..",
        PATH_INFERENCE,
        f"{prefix}lfm-g-{lfm_global}_lfm-t-{lfm_tc}_d-{decade}/")

def create_posterior_file_name(folder_name: str) -> str:
    return os.path.join(folder_name, "posteriors.npz")

In [ ]:
np_file = np.load(create_posterior_file_name(create_folder_name(lfm_global=CompoundLFM.PAH.value, lfm_tc=CompoundLFM.PAH.value, decade=1990)))
print(np_file.files)

In [ ]:
# Iterate over decades from 1970 to 2000
decades = range(1970, 2010, 10)
results_mean = {combo: [] for combo in LFM_COMBS}
results_std = {combo: [] for combo in LFM_COMBS}
for decade in decades:
    for lfm_global, lfm_tc in LFM_COMBS:
        file_name = create_file_name(lfm_global=lfm_global, lfm_tc=lfm_tc, decade=decade)
        with np.load(file_name) as data:
            results_mean[(lfm_global, lfm_tc)].append(np.mean(data['discrepancies']))
            results_std[(lfm_global, lfm_tc)].append(np.std(data['discrepancies']))

# Plotting
plt.figure(figsize=(10, 6))
for (lfm_global, lfm_tc), values in results_mean.items():
    if any(v is not None for v in values):
        plt.errorbar(decades, [v if v is not None else float('nan') for v in values], yerr=[v if v is not None else float('nan') for v in results_std[(lfm_global, lfm_tc)]], fmt='o-', label=f"{lfm_global} - {lfm_tc}")
plt.xlabel('Decade')
plt.ylabel('Mean Discrepancies')
plt.title('Mean Discrepancies over Decades for Different LFM Combinations')
plt.legend()
plt.show()


In [ ]:
# Create a grid of subplots
fig, axes = plt.subplots(
    len(LFM_COMBS), len(decades),
    sharex=True, sharey=True)
bins = np.linspace(0, 1., 25)
# Iterate over LFM combinations and decades to plot heatmaps
for i, (lfm_global, lfm_tc) in enumerate(LFM_COMBS):
    for j, decade in enumerate(decades):
        file_name = create_file_name(lfm_global=lfm_global, lfm_tc=lfm_tc, decade=decade)
        if os.path.exists(file_name):
            with np.load(file_name) as data:
                h = data['h']
                tau = data['tau']
                heatmap_data, _, _ = np.histogram2d(h, tau, density=True, bins=bins)
                axes[i, j].imshow(heatmap_data, cmap="viridis", origin='lower', extent=(0, 1, 0, 1))

for ax, decade in zip(axes[0], decades):
    ax.set_title(f"{decade}", fontsize=10)
for ax in axes[:, 0]:
    ax.set_ylabel(r"$h$")
for ax in axes[-1]:
    ax.set_xlabel(r"$\tau$")
for ax, (lfm_global, lfm_tc) in zip(axes[:, 0], LFM_COMBS):
    ax.text(
        -.15, 1.15,
        f"{MAP_LFM_SHORT[lfm_global]}-{MAP_LFM_SHORT[lfm_tc]}",
        va='center',
        transform=ax.transAxes)

plt.tight_layout()
plt.show()

## Summary statistics

In [ ]:
FOLDER_ARRAY_POOL = "summary_stats_sim"
# L_METRICS = ["ccf_0", "ccf_1", "ccf_2", "ccf_3", "ccf_4", "ccf_5", "ccf_6", "ccf_7", "ei", "gini", "gini_maj", "gini_min", "mann_whitney"]
L_METRICS = ["mean_ccf", "ei", "gini", "gini_maj", "gini_min", "mann_whitney"]
# L_METRICS = ["ccf_0", "ccf_3", "ccf_6", "ccf_7", "ei", "gini", "gini_maj", "gini_min", "mann_whitney"]

In [ ]:
d_metrics = {
    metric: np.load(
        os.path.join(create_folder_name(
            lfm_global=CompoundLFM.PAH.value,
            lfm_tc=CompoundLFM.PAH.value), FOLDER_ARRAY_POOL, f"{metric}.npy"))\
    for metric in L_METRICS}

In [ ]:
a_posteriors = np.load(
    os.path.join(create_folder_name(
        lfm_global=CompoundLFM.PAH.value,
        lfm_tc=CompoundLFM.PAH.value), "posteriors.npz")
)
print("Observed metrics:")
for metric in L_METRICS:
    print(f"{metric}_obs: {a_posteriors[metric]}")

In [ ]:
print("Mean and std of simulations:")
for metric, data in d_metrics.items():
    print(f"{metric}: {np.mean(data)}, {np.std(data)}")

In [ ]:
# Plot metric distributions in a grid
fig, axes = plt.subplots(
    3, 5, figsize=(20, 12))
for i, metric in enumerate(L_METRICS):
    ax = axes.flatten()[i]
    data = d_metrics[metric]
    ax.hist(data, bins=25, density=True)
    ax.axvline(a_posteriors[metric], color='r', linestyle='--')
    ax.set_title(metric)
fig.tight_layout()